# Enformer Regulatory Activity Prediction

Enformer is a deep learning model for predicting gene regulatory activity directly from DNA sequence, published by Avsec et al. (2021). It processes 196,608 bp input windows and outputs binned predictions across 896 spatial bins for thousands of regulatory tracks covering chromatin accessibility, histone modifications, and gene expression. The model supports both human and mouse output heads, and track selection lets you extract only the signals relevant to your analysis.

This notebook demonstrates how to predict regulatory activity from a DNA sequence and how to perform a variant-effect comparison by running reference and alternate allele windows through the model and computing log2 fold-change.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("enformer")
display_overview("enformer")
display_docs_section("enformer", "Background")

# Enformer

Enformer is a transformer-based deep learning model that predicts gene expression and chromatin accessibility directly from a 196,608 bp DNA sequence. It uses [self-attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) to capture long-range regulatory interactions up to ~100 kb away, enabling accurate prediction of how distal enhancers, silencers, and other regulatory elements influence gene expression.

- **Tool key**: `enformer-prediction`
- **Model context**: 196,608 bp (fixed length, ~98 kb in each direction from center)
- **Output resolution**: 896 bins x 128 bp per bin
- **Species heads**: `human` (5,313 tracks), `mouse` (1,643 tracks)
- **GPU required**: Yes

## Background

Gene expression is controlled by a complex interplay of promoters, [enhancers](https://en.wikipedia.org/wiki/Enhancer_(genetics)), [silencers](https://en.wikipedia.org/wiki/Silencer_(genetics)), [insulators](https://en.wikipedia.org/wiki/Insulator_(genetics)), and chromatin state. These regulatory elements can act over distances of tens to hundreds of kilobases. Traditional motif-based models capture local sequence features but miss long-range interactions.

Enformer addresses this by processing 196,608 bp of genomic context through a transformer architecture. The model was trained on 5,313 human and 1,643 mouse experimental tracks from [ENCODE](https://en.wikipedia.org/wiki/ENCODE) and Roadmap Epigenomics, covering:
- **Gene expression**: [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_of_gene_expression) (promoter-level transcription initiation)
- **Chromatin accessibility**: [DNase-seq](https://en.wikipedia.org/wiki/DNase-Seq), [ATAC-seq](https://en.wikipedia.org/wiki/ATAC-seq)
- **Histone modifications**: H3K4me3, H3K27ac, H3K36me3, H3K27me3, and others
- **Transcription factor binding**: [ChIP-seq](https://en.wikipedia.org/wiki/ChIP_sequencing) for hundreds of TFs

Each output track corresponds to a specific assay in a specific cell type or tissue. The 896 output bins (128 bp each) tile the central ~114 kb of the input window, providing spatial resolution of predicted regulatory activity.

## Available tools

In [2]:
display_available_tools("enformer")

- **`run_enformer()`** — Gene expression and regulatory activity prediction using Enformer

### `run_enformer`

Enformer predicts regulatory activity from a 196,608 bp DNA input window, returning a signal matrix of shape [896, num_tracks] where each of the 896 spatial bins covers a 128 bp window. The `output_tracks` parameter selects which of the thousands of regulatory tracks to extract, and the `species` parameter switches between human and mouse output heads. A common use is variant-effect analysis: running the same input window with reference and alternate alleles and comparing the resulting prediction matrices to identify regulatory impact.

In [3]:
import numpy as np
from proto_tools import ENFORMER_CONTEXT, EnformerConfig, EnformerInput, run_enformer

In [4]:
# Display input docs
display_api_reference("enformer", "input", "run_enformer")

# Create a full-length DNA sequence by repeating a short pattern to fill the required context
sequence = "ATCG" * (ENFORMER_CONTEXT // 4)
inputs = EnformerInput(sequences=[sequence])

**Input** — `EnformerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[proto_tools.tools.sequence_scoring.shared_data_models.SequenceWindow]` | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [5]:
# Display config docs
display_api_reference("enformer", "config", "run_enformer")

config = EnformerConfig(
    output_tracks=[0, 1, 2],
    species="human",
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `EnformerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `output_tracks` | `list[int]` | `[0]` | Indices of Enformer output tracks to extract (5313 human / 1643 mouse) |
| `species` | `Literal['human', 'mouse']` | `'human'` | Species track head to use |
| `batch_size` | `int` | `1` | Number of sequences to process simultaneously on GPU |

In [6]:
# Run the prediction function
pred_result = run_enformer(inputs, config)

Running run_enformer [00:00]

In [7]:
display_api_reference("enformer", "output", "run_enformer")

print(f"Output bins: {len(pred_result.results[0].prediction)}")
print(f"Tracks per bin: {len(pred_result.results[0].prediction[0])}")

**Output** — `EnformerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.sequence_scoring.enformer.enformer_prediction.EnformerPredictionResult]` | required | Per-sequence Enformer prediction results |
| `output_tracks` | `list[int]` | required | Track indices extracted from Enformer |
| `species` | `str` | required | Species used for prediction ('human' or 'mouse') |

Output bins: 896
Tracks per bin: 3


#### Variant-Effect Comparison

A common use of Enformer is comparing predictions between a reference and an alternate allele to identify regulatory variants. We introduce a single-nucleotide change at the center of the input window, run both sequences through the model, and compute log2 fold-change across all bins and tracks. Large absolute fold-change values indicate positions and tracks where the variant has a meaningful effect on predicted regulatory activity.

In [8]:
# Introduce a single-nucleotide substitution at the center of the input window
center = ENFORMER_CONTEXT // 2
ref_sequence = sequence
alt_base = "A" if ref_sequence[center] != "A" else "C"
alt_sequence = ref_sequence[:center] + alt_base + ref_sequence[center + 1:]

ref_result = run_enformer(EnformerInput(sequences=[ref_sequence]), config)
alt_result = run_enformer(EnformerInput(sequences=[alt_sequence]), config)

ref_pred = np.array(ref_result.results[0].prediction)
alt_pred = np.array(alt_result.results[0].prediction)
log2_fc = np.log2((alt_pred + 1e-6) / (ref_pred + 1e-6))
print(f"Max |log2 FC|: {np.abs(log2_fc).max():.4f}")

Running run_enformer [00:00]

Running run_enformer [00:00]

Max |log2 FC|: 0.0962


## Export Results

Prediction outputs can be saved to standard file formats for downstream analysis or integration with other computational pipelines. Supported formats include JSON and CSV.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

pred_result.export(name="enformer_prediction", export_path=output_dir, file_format="json")
print(f"Prediction exported to {output_dir}/enformer_prediction.json")

ref_result.export(name="enformer_ref", export_path=output_dir, file_format="csv")
print(f"Reference summary exported to {output_dir}/enformer_ref.csv")

Prediction exported to example_output/enformer_prediction.json
Reference summary exported to example_output/enformer_ref.csv
